# Phase 4: Feature Engineering (Batch)

In this notebook, we'll engineer the features required for training our ML models. 

We will compute:
1. **Static/Batch Features**: Mathematical relationships (e.g. balance drops, ratios) and temporal data (hour of day).
2. **Historical Windowed Features**: To train the models to recognize velocity, we need to approximate the real-time Faust windowed features for our historical dataset.

Finally, we'll save the result as a parquet file (`data/processed/features.parquet`) to be ingested by the training scripts.

In [2]:
import pandas as pd
import numpy as np
import warnings
import os
import sys

warnings.filterwarnings('ignore')

# Add project root to python path to allow importing 'features' module
if '..' not in sys.path:
    sys.path.append('..')

# Ensure the processed directory exists
os.makedirs('../data/processed', exist_ok=True)

print("Loading raw dataset...")
df = pd.read_csv('../data/raw/paysim_dataset.csv')
print(f"Loaded {len(df):,} rows.")

Loading raw dataset...
Loaded 6,362,620 rows.


### 1. Compute Static/Batch Features
These are per-transaction features that can be computed instantly from the transaction's own data.

In [3]:
print("Computing static features...")

# 1. balance_drop_pct
df['balance_drop_pct'] = (df['oldbalanceOrg'] - df['newbalanceOrig']) / (df['oldbalanceOrg'] + 1.0)

# 2. amount_to_balance_ratio
df['amount_to_balance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1.0)

# 3. is_large_transfer
df['is_large_transfer'] = ((df['type'] == 'TRANSFER') & (df['amount'] > 200000)).astype(int)

# 4. dest_balance_increased
df['dest_balance_increased'] = (df['newbalanceDest'] > df['oldbalanceDest']).astype(int)

# 5. Time features (step = 1 hour)
df['hour_of_day'] = df['step'] % 24
df['day_of_month'] = df['step'] // 24

# 6. One-hot encode `type`
print("One-hot encoding 'type' column...")
df = pd.get_dummies(df, columns=['type'], prefix='type')

# Ensure all expected one-hot columns exist (in case dataset was subsetted)
expected_types = ['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']
for t in expected_types:
    if t not in df.columns:
        df[t] = 0
    df[t] = df[t].astype(int) # Convert boolean dummies to int

print("Static features computed.")

Computing static features...
One-hot encoding 'type' column...
Static features computed.


### 2. Compute Historical Windowed Features
In production, Faust computes these in real-time over 5m, 1h, and 24h rolling windows using actual wall-clock time.

For historical training on PaySim, `step` is only granular to 1 hour. We will approximate these historical windows by grouping by `nameOrig` and `step`.

In [4]:
print("Computing historical windowed features...")

# Sort by time and account to allow rolling operations
df = df.sort_values(by=['nameOrig', 'step']).reset_index(drop=True)

# Since 'step' is 1 hour, a 5-minute window is impossible to construct exactly.
# We will use the current step's rolling count to approximate velocity features.
# For 24h, we look back 24 steps.

def compute_windows(group):
    group = group.set_index('step')
    
    # 1h features (in PaySim, everything in the same step is effectively a 1h window)
    # We'll calculate cumulative stats within the same step
    group['txn_count_1h'] = group.groupby(level=0).cumcount() + 1
    group['avg_amount_1h'] = group['amount'].groupby(level=0).expanding().mean().reset_index(level=0, drop=True)
    group['max_amount_1h'] = group['amount'].groupby(level=0).expanding().max().reset_index(level=0, drop=True)
    
    # Approximation for 5m (just capturing immediate bursts)
    group['txn_count_5m'] = group['txn_count_1h'] 
    
    # 24h feature (rolling over the last 24 steps)
    # Reindex to fill missing steps with 0 counts, then rolling sum
    counts = group.groupby(level=0).size()
    rolling_24h = counts.reindex(range(counts.index.min(), counts.index.max() + 1), fill_value=0).rolling(24, min_periods=1).sum()
    group['txn_count_24h'] = group.index.map(rolling_24h)
    
    # Unique dest 1h
    group['unique_dest_1h'] = group.groupby(level=0)['nameDest'].transform(lambda s: (~s.duplicated()).cumsum())
    
    return group.reset_index()

print("Applying rolling windows per account (this may take 2-3 minutes for 6.3M rows)...")
# Optimization: Only apply rolling logic to accounts with more than 1 transaction to save massive computation time.
account_counts = df['nameOrig'].value_counts()
multi_txn_accounts = account_counts[account_counts > 1].index

# For accounts with only 1 transaction, windowed features are deterministic
single_txn_mask = ~df['nameOrig'].isin(multi_txn_accounts)
df.loc[single_txn_mask, 'txn_count_5m'] = 1
df.loc[single_txn_mask, 'txn_count_1h'] = 1
df.loc[single_txn_mask, 'avg_amount_1h'] = df.loc[single_txn_mask, 'amount']
df.loc[single_txn_mask, 'max_amount_1h'] = df.loc[single_txn_mask, 'amount']
df.loc[single_txn_mask, 'unique_dest_1h'] = 1
df.loc[single_txn_mask, 'txn_count_24h'] = 1

# Now process the multi-transaction accounts
if len(multi_txn_accounts) > 0:
    multi_df = df[df['nameOrig'].isin(multi_txn_accounts)]
    multi_df = multi_df.groupby('nameOrig').apply(compute_windows).reset_index(drop=True)
    
    # Merge back
    df = pd.concat([df[single_txn_mask], multi_df], ignore_index=True)

print("Windowed features computed.")

Computing historical windowed features...
Applying rolling windows per account (this may take 2-3 minutes for 6.3M rows)...
Windowed features computed.


### 3. Cleanup and Save
Drop unneeded ID strings and sort the columns to match `ALL_FEATURE_NAMES`.
We also save a small CSV for manual human inspection!

In [5]:
from features.feature_definitions import ALL_FEATURE_NAMES

# 1. Save the full Parquet file (Models need this to load quickly!)
keep_columns = ['step', 'isFraud'] + ALL_FEATURE_NAMES
print("Selecting required columns for ML...")
df_final = df[keep_columns].copy()

print("Checking for nulls:")
print(df_final.isnull().sum())

print("\nFinal ML Dataset Shape:", df_final.shape)

out_path = '../data/processed/features.parquet'
print(f"\nSaving ML dataset to {out_path} ...")
df_final.to_parquet(out_path, engine='pyarrow', index=False)

# 2. Save a CSV version with all raw columns for human inspection!
# We will save a random sample of 10,000 rows. A full 6.3M row CSV would be ~2GB and crash your text editor.
csv_path = '../data/processed/features_inspection.csv'
print(f"\nSaving a 10,000-row sample to {csv_path} for easy viewing...")
df.sample(n=10000, random_state=42).to_csv(csv_path, index=False)

print("✅ Done! Ready for Phase 5 (Model Training).")

Selecting required columns for ML...
Checking for nulls:
step                       0
isFraud                    0
txn_count_5m               0
txn_count_1h               0
avg_amount_1h              0
max_amount_1h              0
unique_dest_1h             0
balance_drop_pct           0
txn_count_24h              0
amount_to_balance_ratio    0
is_large_transfer          0
dest_balance_increased     0
hour_of_day                0
day_of_month               0
amount                     0
oldbalanceOrg              0
newbalanceOrig             0
oldbalanceDest             0
newbalanceDest             0
type_CASH_IN               0
type_CASH_OUT              0
type_DEBIT                 0
type_PAYMENT               0
type_TRANSFER              0
dtype: int64

Final ML Dataset Shape: (6362620, 24)

Saving ML dataset to ../data/processed/features.parquet ...

Saving a 10,000-row sample to ../data/processed/features_inspection.csv for easy viewing...
✅ Done! Ready for Phase 5 (Model Training